In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
model = YOLO('yolov10n.pt')

In [ ]:
def get_traffic_light_color(img):
    """Simple HSV-based color detection for traffic lights."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Define color ranges (Simplified)
    # Note: These ranges may need tuning based on lighting conditions
    ranges = {
        'Red': ([0, 100, 100], [10, 255, 255]),
        'Yellow': ([20, 100, 100], [30, 255, 255]),
        'Green': ([40, 100, 100], [90, 255, 255])
    }
    
    max_pixels = 0
    detected_color = "Unknown"
    
    for color, (lower, upper) in ranges.items():
        mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
        count = cv2.countNonZero(mask)
        if count > max_pixels:
            max_pixels = count
            detected_color = color
            
    return detected_color

In [ ]:
results = model('images_to_check/1.png')[0]


image 1/1 /home/rutwik/Documents/learn/traffic lights/images_to_check/1.png: 640x416 1 traffic light, 32.8ms
Speed: 1.9ms preprocess, 32.8ms inference, 8.0ms postprocess per image at shape (1, 3, 640, 416)


In [ ]:
traffic_lights = []
for box in results.boxes:
    # Class 9 is usually 'traffic light' in COCO dataset
    if int(box.cls) == 9:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        area = (x2 - x1) * (y2 - y1)
        traffic_lights.append({'box': (x1, y1, x2, y2), 'area': area})

if traffic_lights:
    # Select the "nearest" (largest area)
    nearest = max(traffic_lights, key=lambda x: x['area'])
    x1, y1, x2, y2 = nearest['box']
    
    # Crop to the detected light
    cropped_light = results.orig_img[y1:y2, x1:x2]
    
    # Detect color
    color = get_traffic_light_color(cropped_light)
    print(f"Nearest Traffic Light Detected at [{x1}, {y1}, {x2}, {y2}]")
    print(f"Status: {color}")
else:
    print("No traffic lights detected.")

Nearest Traffic Light Detected at [87, 96, 400, 720]
Status: Green
